# EfficientNet with Stochastic Weight Averaging (SWA)

This notebook implements Stochastic Weight Averaging:
- Averages model weights from different training stages
- Finds flatter minima in the loss landscape
- Improves generalization without extra computation
- Often achieves better performance than the best single model

In [6]:
from torch import optim
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch
import random
import numpy as np
import torch.nn as nn
import albumentations as Albu
import pandas as pd
from torch.utils.data.sampler import RandomSampler
from warmup_scheduler import GradualWarmupScheduler
from torch.optim.swa_utils import AveragedModel, SWALR
from tqdm import tqdm
import os
import sys
sys.path.append('../../..')
from utils.dataset import PandasDataset
from utils.metrics import model_checkpoint, evaluation, format_metrics
from utils.models import EfficientNetApi

In [7]:
seed = 42
batch_size = 6
num_workers = 4
output_classes = 5
init_lr = 3e-4
warmup_factor = 2
warmup_epochs = 1
n_epochs = 50
swa_start = 30  # Start SWA after epoch 30
swa_lr = 1e-4   # SWA learning rate
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
loss_function = nn.BCEWithLogitsLoss()

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

ROOT_DIR = '../../..'
data_dir = '../../../../dataset'
images_dir = os.path.join(data_dir, 'tiles')

Using device: cuda


In [8]:
load_model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
model = EfficientNetApi(model=load_model, output_dimensions=output_classes, dropout_rate=0.6)
model = model.to(device)

# Create SWA model
swa_model = AveragedModel(model)
print("Model and SWA model created")

Model and SWA model created


## Load Dataset

In [9]:
df_train_ = pd.read_csv(f"{ROOT_DIR}/data/train_5fold.csv")
df_train_.columns = df_train_.columns.str.strip()
train_indexes = np.where((df_train_['fold'] != 3))[0]
valid_indexes = np.where((df_train_['fold'] == 3))[0]

df_train = df_train_.loc[train_indexes]
df_val = df_train_.loc[valid_indexes]
df_test = pd.read_csv(f"{ROOT_DIR}/data/test.csv")

transforms = Albu.Compose([
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
])

train_dataset = PandasDataset(images_dir, df_train, transforms=transforms)
valid_dataset = PandasDataset(images_dir, df_val, transforms=None)
test_dataset = PandasDataset(images_dir, df_test, transforms=None)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, num_workers=num_workers, sampler=RandomSampler(train_dataset)
)
valid_loader = torch.utils.data.DataLoader(
    valid_dataset, batch_size=batch_size, num_workers=num_workers, sampler=RandomSampler(valid_dataset)
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=batch_size, num_workers=num_workers, sampler=RandomSampler(test_dataset)
)

## Custom Training Loop with SWA

In [11]:
from utils.train import train_model

optimizer = optim.Adam(model.parameters(), lr=init_lr/warmup_factor)
scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs - warmup_epochs)
scheduler = GradualWarmupScheduler(optimizer, multiplier=warmup_factor, total_epoch=warmup_epochs, after_scheduler=scheduler_cosine)

# SWA scheduler
swa_scheduler = SWALR(optimizer, swa_lr=swa_lr)

train_model(
    model=model,
    epochs=n_epochs,
    optimizer=optimizer,
    scheduler=scheduler,
    train_dataloader=train_loader,
    valid_dataloader=valid_loader,
    checkpoint=model_checkpoint,
    device=device,
    loss_function=loss_function,
    path_to_save_metrics="logs/swa.txt",
    path_to_save_model="models/swa-regular.pth",
    patience=5,
)

Epoch 1/50



100%|██████████| 301/301 [01:39<00:00,  3.02it/s]


VAL_LOSS     0.288
VAL_ACC      Mean: 54.800 | Std: 1.179 | 95% CI: [52.909, 56.789]
VAL_KAPPA    Mean: 0.768 | Std: 0.012 | 95% CI: [0.748, 0.787]
VAL_F1       Mean: 0.445 | Std: 0.012 | 95% CI: [0.425, 0.465]
VAL_RECALL   Mean: 0.449 | Std: 0.011 | 95% CI: [0.431, 0.467]
VAL_PRECISION Mean: 0.552 | Std: 0.013 | 95% CI: [0.530, 0.575]
Salvando o melhor modelo... 0.0 -> 0.768419097844179
Epoch 2/50



100%|██████████| 301/301 [01:34<00:00,  3.19it/s]
/home/woshington/Projects/Doutorado/repo/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1087: UserWarning: To get the last learning rate computed by the scheduler, please use `get_last_lr()`.
  _warn_get_lr_called_within_step(self)


VAL_LOSS     0.329
VAL_ACC      Mean: 53.051 | Std: 1.191 | 95% CI: [51.080, 54.958]
VAL_KAPPA    Mean: 0.751 | Std: 0.013 | 95% CI: [0.729, 0.773]
VAL_F1       Mean: 0.429 | Std: 0.012 | 95% CI: [0.411, 0.449]
VAL_RECALL   Mean: 0.437 | Std: 0.011 | 95% CI: [0.420, 0.454]
VAL_PRECISION Mean: 0.542 | Std: 0.012 | 95% CI: [0.522, 0.561]
Epoch 3/50



...

VAL_LOSS     0.488
VAL_ACC      Mean: 63.039 | Std: 1.156 | 95% CI: [61.163, 64.931]
VAL_KAPPA    Mean: 0.823 | Std: 0.012 | 95% CI: [0.803, 0.842]
VAL_F1       Mean: 0.571 | Std: 0.013 | 95% CI: [0.550, 0.591]
VAL_RECALL   Mean: 0.570 | Std: 0.012 | 95% CI: [0.550, 0.589]
VAL_PRECISION Mean: 0.600 | Std: 0.013 | 95% CI: [0.580, 0.621]
Epoch 13/50



100%|██████████| 301/301 [01:33<00:00,  3.24it/s]


VAL_LOSS     0.502
VAL_ACC      Mean: 63.884 | Std: 1.122 | 95% CI: [62.161, 65.817]
VAL_KAPPA    Mean: 0.819 | Std: 0.012 | 95% CI: [0.799, 0.839]
VAL_F1       Mean: 0.581 | Std: 0.012 | 95% CI: [0.561, 0.601]
VAL_RECALL   Mean: 0.576 | Std: 0.012 | 95% CI: [0.558, 0.596]
VAL_PRECISION Mean: 0.615 | Std: 0.013 | 95% CI: [0.594, 0.636]
Epoch 14/50



100%|██████████| 301/301 [01:29<00:00,  3.36it/s]


VAL_LOSS     0.504
VAL_ACC      Mean: 63.727 | Std: 1.112 | 95% CI: [61.884, 65.540]
VAL_KAPPA    Mean: 0.823 | Std: 0.012 | 95% CI: [0.802, 0.842]
VAL_F1       Mean: 0.581 | Std: 0.012 | 95% CI: [0.562, 0.601]
VAL_RECALL   Mean: 0.579 | Std: 0.012 | 95% CI: [0.560, 0.598]
VAL_PRECISION Mean: 0.599 | Std: 0.012 | 95% CI: [0.581, 0.619]

Early stopping at epoch 14. No improvement for 5 epochs.
Best epoch: 9 with kappa: 0.8259


## Test Regular Model

In [12]:
model.load_state_dict(torch.load("models/swa-regular.pth"))
response = evaluation(model, test_loader, device)
result = format_metrics(response[0])
print("\n=== TEST RESULTS (Regular Model) ===")
print(result)

100%|██████████| 266/266 [01:15<00:00,  3.54it/s]



=== TEST RESULTS (Regular Model) ===
VAL_ACC      Mean: 61.781 | Std: 1.182 | 95% CI: [59.862, 63.693]
VAL_KAPPA    Mean: 0.837 | Std: 0.013 | 95% CI: [0.815, 0.857]
VAL_F1       Mean: 0.555 | Std: 0.013 | 95% CI: [0.533, 0.575]
VAL_RECALL   Mean: 0.554 | Std: 0.012 | 95% CI: [0.533, 0.574]
VAL_PRECISION Mean: 0.565 | Std: 0.013 | 95% CI: [0.543, 0.586]
